# 🖼️ 이미지 전처리 (Image Preprocessing)

> **학습 목표**: 딥러닝 모델 학습을 위한 이미지 전처리 기법을 습득합니다.

---

## 📋 학습 내용

1. ✅ 이미지 로드 및 기본 정보 확인
2. ✅ 크기 조정 (Resize, Crop)
3. ✅ 정규화 (Normalization) - [0,1] 범위
4. ✅ 데이터 증강 (Augmentation) - 회전, 반전, 밝기
5. ✅ 노이즈 제거 (Gaussian, Median Filter)
6. ✅ Edge Detection (Canny, Sobel)

**소요 시간**: 약 35분  
**난이도**: ⭐⭐ (중급)  
**사전 지식**: Python 기초, 이미지 개념

---

## 🔧 Step 1: 라이브러리 Import

In [ ]:
# 이미지 처리
from PIL import Image, ImageEnhance, ImageFilter
import cv2

# 데이터 분석
import numpy as np
import pandas as pd

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 유틸리티
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ 라이브러리 로드 완료!")
print(f"   PIL 버전: {Image.__version__}")
print(f"   OpenCV 버전: {cv2.__version__}")

## 📂 Step 2: 이미지 데이터 로드

In [ ]:
# 이미지 데이터 경로
data_dir = Path('../data/defect_images')

# 이미지 파일 목록
image_files = list(data_dir.glob('*.jpg')) + list(data_dir.glob('*.png'))

if len(image_files) == 0:
    print(f"⚠️ 이미지 파일이 없습니다: {data_dir}")
    print("   샘플 이미지를 생성합니다...")
    
    # 샘플 이미지 생성 (256x256 그레이스케일)
    data_dir.mkdir(exist_ok=True, parents=True)
    
    for i in range(5):
        # 랜덤 노이즈 이미지 생성
        img_array = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        img.save(data_dir / f'sample_{i:02d}.png')
    
    image_files = list(data_dir.glob('*.png'))
    print(f"✅ 샘플 이미지 {len(image_files)}개 생성")
else:
    print(f"✅ 이미지 파일 발견: {len(image_files)}개")

print(f"\n📊 이미지 목록 (처음 10개):")
for img_path in image_files[:10]:
    print(f"   - {img_path.name}")

In [ ]:
# 첫 번째 이미지 로드 및 정보 확인
sample_img_path = image_files[0]
img = Image.open(sample_img_path)

print(f"🖼️ 이미지 정보:")
print(f"   파일: {sample_img_path.name}")
print(f"   크기: {img.size} (width x height)")
print(f"   모드: {img.mode}")
print(f"   포맷: {img.format}")

# NumPy 배열로 변환
img_array = np.array(img)
print(f"\n📊 NumPy 배열 정보:")
print(f"   형태: {img_array.shape}")
print(f"   데이터 타입: {img_array.dtype}")
print(f"   값 범위: [{img_array.min()}, {img_array.max()}]")

# 원본 이미지 시각화
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.title(f'원본 이미지: {sample_img_path.name}', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 📏 Step 3: 크기 조정 (Resize & Crop)

In [ ]:
# 1. Resize (크기 변경)
target_size = (224, 224)  # 대부분의 CNN 모델 입력 크기

img_resized = img.resize(target_size, Image.LANCZOS)

print(f"✅ Resize 완료: {img.size} → {img_resized.size}")

# 2. Center Crop (중앙 자르기)
crop_size = (200, 200)

width, height = img_resized.size
left = (width - crop_size[0]) // 2
top = (height - crop_size[1]) // 2
right = left + crop_size[0]
bottom = top + crop_size[1]

img_cropped = img_resized.crop((left, top, right, bottom))

print(f"✅ Center Crop 완료: {img_resized.size} → {img_cropped.size}")

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img)
axes[0].set_title(f'원본\n{img.size}', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_resized)
axes[1].set_title(f'Resize\n{img_resized.size}', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(img_cropped)
axes[2].set_title(f'Center Crop\n{img_cropped.size}', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 🔢 Step 4: 정규화 (Normalization)

In [ ]:
# NumPy 배열로 변환
img_array = np.array(img_resized)

# 1. Min-Max Normalization [0, 1]
img_normalized = img_array / 255.0

print(f"📊 정규화 전:")
print(f"   데이터 타입: {img_array.dtype}")
print(f"   값 범위: [{img_array.min()}, {img_array.max()}]")
print(f"   평균: {img_array.mean():.2f}")
print(f"   표준편차: {img_array.std():.2f}")

print(f"\n📊 정규화 후 [0, 1]:")
print(f"   데이터 타입: {img_normalized.dtype}")
print(f"   값 범위: [{img_normalized.min():.4f}, {img_normalized.max():.4f}]")
print(f"   평균: {img_normalized.mean():.4f}")
print(f"   표준편차: {img_normalized.std():.4f}")

# 2. Z-Score Standardization (평균 0, 표준편차 1)
mean = img_array.mean(axis=(0, 1), keepdims=True)
std = img_array.std(axis=(0, 1), keepdims=True)
img_standardized = (img_array - mean) / (std + 1e-7)

print(f"\n📊 표준화 후 (Z-Score):")
print(f"   값 범위: [{img_standardized.min():.4f}, {img_standardized.max():.4f}]")
print(f"   평균: {img_standardized.mean():.4f}")
print(f"   표준편차: {img_standardized.std():.4f}")

# 시각화 (클리핑 필요)
img_standardized_clipped = np.clip((img_standardized - img_standardized.min()) / 
                                    (img_standardized.max() - img_standardized.min()), 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_array.astype(np.uint8))
axes[0].set_title('원본 [0, 255]', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_normalized)
axes[1].set_title('Min-Max [0, 1]', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(img_standardized_clipped)
axes[2].set_title('Z-Score (시각화용)', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 🔄 Step 5: 데이터 증강 (Data Augmentation)

In [ ]:
# 1. 회전 (Rotation)
img_rotated_90 = img_resized.rotate(90, expand=True)
img_rotated_45 = img_resized.rotate(45, fillcolor=(255, 255, 255))

# 2. 좌우 반전 (Horizontal Flip)
img_flipped_lr = img_resized.transpose(Image.FLIP_LEFT_RIGHT)

# 3. 상하 반전 (Vertical Flip)
img_flipped_tb = img_resized.transpose(Image.FLIP_TOP_BOTTOM)

# 4. 밝기 조정 (Brightness)
enhancer_brightness = ImageEnhance.Brightness(img_resized)
img_brighter = enhancer_brightness.enhance(1.5)  # 1.5배 밝게
img_darker = enhancer_brightness.enhance(0.5)    # 0.5배 어둡게

# 5. 대비 조정 (Contrast)
enhancer_contrast = ImageEnhance.Contrast(img_resized)
img_high_contrast = enhancer_contrast.enhance(1.8)

# 6. 색상 조정 (Color/Saturation)
enhancer_color = ImageEnhance.Color(img_resized)
img_saturated = enhancer_color.enhance(1.5)

print("✅ 데이터 증강 완료!")
print(f"   원본: 1개")
print(f"   증강: 8개 (회전 2개, 반전 2개, 밝기 2개, 대비 1개, 색상 1개)")
print(f"   총: 9개 (9배 증가)")

In [ ]:
# 데이터 증강 시각화
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

augmented_images = [
    (img_resized, '원본'),
    (img_rotated_90, '회전 90°'),
    (img_rotated_45, '회전 45°'),
    (img_flipped_lr, '좌우 반전'),
    (img_flipped_tb, '상하 반전'),
    (img_brighter, '밝게 (1.5x)'),
    (img_darker, '어둡게 (0.5x)'),
    (img_high_contrast, '대비 증가'),
    (img_saturated, '채도 증가')
]

for idx, (aug_img, title) in enumerate(augmented_images):
    row = idx // 3
    col = idx % 3
    axes[row, col].imshow(aug_img)
    axes[row, col].set_title(title, fontsize=12, fontweight='bold')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("\n💡 데이터 증강 활용:")
print("   - 학습 데이터 부족 문제 해결")
print("   - 과적합(Overfitting) 방지")
print("   - 다양한 변형에 대한 강건성(Robustness) 향상")

## 🔊 Step 6: 노이즈 제거 (Denoising)

In [ ]:
# 노이즈 추가 (테스트용)
img_array = np.array(img_resized)

# Salt-and-Pepper 노이즈 추가
noise = np.random.random(img_array.shape[:2])
img_noisy = img_array.copy()
img_noisy[noise < 0.02] = 0      # Salt (검은 점)
img_noisy[noise > 0.98] = 255    # Pepper (흰 점)
img_noisy_pil = Image.fromarray(img_noisy.astype(np.uint8))

# 1. Gaussian Blur (가우시안 블러)
img_gaussian = img_noisy_pil.filter(ImageFilter.GaussianBlur(radius=2))

# 2. Median Filter (중간값 필터)
img_median = img_noisy_pil.filter(ImageFilter.MedianFilter(size=3))

# 3. OpenCV Bilateral Filter (경계 보존 노이즈 제거)
img_bilateral = cv2.bilateralFilter(img_noisy, d=9, sigmaColor=75, sigmaSpace=75)
img_bilateral_pil = Image.fromarray(img_bilateral)

print("✅ 노이즈 제거 완료!")
print(f"   Gaussian Blur: 부드러운 블러링")
print(f"   Median Filter: Salt-and-Pepper 노이즈 제거")
print(f"   Bilateral Filter: 경계 보존하며 노이즈 제거")

In [ ]:
# 노이즈 제거 시각화
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(img_noisy_pil)
axes[0, 0].set_title('노이즈 추가 (Salt-and-Pepper)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_gaussian)
axes[0, 1].set_title('Gaussian Blur', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(img_median)
axes[1, 0].set_title('Median Filter', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_bilateral_pil)
axes[1, 1].set_title('Bilateral Filter', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\n💡 노이즈 제거 비교:")
print("   - Gaussian: 가장 빠르지만 경계도 흐릿해짐")
print("   - Median: Salt-and-Pepper 노이즈에 효과적")
print("   - Bilateral: 경계 보존하며 노이즈 제거 (가장 느림)")

## 🔍 Step 7: Edge Detection (경계 검출)

In [ ]:
# 그레이스케일 변환 (Edge Detection은 보통 그레이스케일에서 수행)
img_gray = cv2.cvtColor(np.array(img_resized), cv2.COLOR_RGB2GRAY)

# 1. Sobel Edge Detection (X, Y 방향)
sobel_x = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_combined = np.sqrt(sobel_x**2 + sobel_y**2)
sobel_combined = np.uint8(255 * sobel_combined / sobel_combined.max())

# 2. Canny Edge Detection
canny_edges = cv2.Canny(img_gray, threshold1=100, threshold2=200)

# 3. Laplacian Edge Detection
laplacian = cv2.Laplacian(img_gray, cv2.CV_64F)
laplacian = np.uint8(np.abs(laplacian))

print("✅ Edge Detection 완료!")
print(f"   Sobel: X, Y 방향 경계 검출")
print(f"   Canny: 최적 경계 검출 (가장 널리 사용)")
print(f"   Laplacian: 2차 미분 기반 경계 검출")

In [ ]:
# Edge Detection 시각화
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(img_gray, cmap='gray')
axes[0, 0].set_title('원본 (그레이스케일)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(sobel_combined, cmap='gray')
axes[0, 1].set_title('Sobel Edge', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(canny_edges, cmap='gray')
axes[1, 0].set_title('Canny Edge', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(laplacian, cmap='gray')
axes[1, 1].set_title('Laplacian Edge', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\n💡 Edge Detection 활용:")
print("   - 객체 경계 추출")
print("   - 불량 검출 (균열, 긁힘, 파손)")
print("   - 피처 추출 (Feature Extraction)")

## 💾 Step 8: 전처리 파이프라인 구축 및 저장

In [ ]:
# 전처리 파이프라인 함수
def preprocess_image(img_path, target_size=(224, 224), normalize=True):
    """
    이미지 전처리 파이프라인
    
    Args:
        img_path: 이미지 파일 경로
        target_size: 목표 크기 (width, height)
        normalize: 정규화 여부 [0, 1]
    
    Returns:
        preprocessed_img: 전처리된 이미지 (NumPy 배열)
    """
    # 1. 로드
    img = Image.open(img_path)
    
    # 2. Resize
    img = img.resize(target_size, Image.LANCZOS)
    
    # 3. NumPy 배열 변환
    img_array = np.array(img)
    
    # 4. 정규화 (선택적)
    if normalize:
        img_array = img_array / 255.0
    
    return img_array

# 전체 이미지 전처리
print("🔄 전체 이미지 전처리 중...")

preprocessed_images = []
for img_path in image_files:
    preprocessed_img = preprocess_image(img_path, target_size=(224, 224), normalize=True)
    preprocessed_images.append(preprocessed_img)

# NumPy 배열로 변환 (batch)
preprocessed_batch = np.array(preprocessed_images)

print(f"✅ 전처리 완료!")
print(f"   배치 크기: {preprocessed_batch.shape}")
print(f"   데이터 타입: {preprocessed_batch.dtype}")
print(f"   값 범위: [{preprocessed_batch.min():.4f}, {preprocessed_batch.max():.4f}]")

In [ ]:
# 출력 디렉토리 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# 전처리 결과 저장
output_file = output_dir / '02_preprocessed_images.npy'
np.save(output_file, preprocessed_batch)
print(f"✅ 전처리 이미지 저장: {output_file}")

# 전처리 통계 저장
stats = pd.DataFrame({
    '항목': ['이미지 개수', '크기 (H×W×C)', '데이터 타입', '최소값', '최대값', '평균', '표준편차'],
    '값': [
        len(preprocessed_images),
        f"{preprocessed_batch.shape[1]}×{preprocessed_batch.shape[2]}×{preprocessed_batch.shape[3]}",
        str(preprocessed_batch.dtype),
        f"{preprocessed_batch.min():.4f}",
        f"{preprocessed_batch.max():.4f}",
        f"{preprocessed_batch.mean():.4f}",
        f"{preprocessed_batch.std():.4f}"
    ]
})

stats_file = output_dir / '02_preprocessing_stats.csv'
stats.to_csv(stats_file, index=False, encoding='utf-8-sig')
print(f"✅ 전처리 통계 저장: {stats_file}")

print("\n🎉 이미지 전처리 완료!")

---

## 🎯 학습 정리

### ✅ 완료한 내용
1. 이미지 로드 및 기본 정보 확인
2. Resize, Center Crop으로 크기 조정
3. Min-Max, Z-Score 정규화
4. 회전, 반전, 밝기/대비/색상 조정으로 데이터 증강
5. Gaussian, Median, Bilateral 필터로 노이즈 제거
6. Sobel, Canny, Laplacian으로 경계 검출
7. 전처리 파이프라인 구축 및 배치 처리

### 💡 핵심 인사이트
- **정규화의 중요성**:
  - [0, 1] 정규화: 딥러닝 모델 학습 안정화
  - Z-Score: 통계적 분석에 유리

- **데이터 증강 효과**:
  - 학습 데이터 9배 증가 (1개 → 9개)
  - 다양한 변형에 대한 강건성 향상
  - 과적합 방지

- **노이즈 제거 선택**:
  - Gaussian: 속도 우선
  - Median: Salt-and-Pepper 노이즈
  - Bilateral: 경계 보존 (품질 우선)

- **Edge Detection 활용**:
  - Canny: 가장 널리 사용 (최적)
  - Sobel: 방향성 정보 필요 시
  - Laplacian: 빠른 경계 검출

### 📚 다음 단계
- **03_defect_classification.ipynb**: CNN 기반 불량 분류
- **Part 2-2**: Vision Transformer (ViT) 활용

### 🔗 참고 자료
- [PIL Documentation](https://pillow.readthedocs.io/)
- [OpenCV Documentation](https://docs.opencv.org/)
- [ImageNet 전처리 가이드](https://github.com/pytorch/vision)

---

*제조AI 교육 v12 Enhanced | Part 1-2 | 2025.02*